# NB02 — Posthoc Gene-Space Evaluation and Biological Validation

This notebook loads the **best trained checkpoint from NB01**, reconstructs predictions into gene space, and computes:

- final overall metrics
- per-cell metrics
- top-k DEG AUPRC
- percentile DEG AUPRC
- pathway profile recovery

**What to send back after running**
- Final overall metrics
- Final per-cell metrics
- DEG AUPRC summary
- Pathway recovery summary


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install anndata scanpy torch pandas scikit-learn scipy

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 139.6 MB/s eta 0:00:00


In [ ]:
import os, random
from dataclasses import dataclass

import anndata as ad
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import average_precision_score, r2_score
from scipy.stats import pearsonr

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

@dataclass
class CFG:
    data_path: str = "/content/drive/MyDrive/ChemDFM/data/sciplex_complete_middle_subset.h5ad"
    ckpt_dir: str = "/content/drive/MyDrive/ChemDFM/checkpoints_nb01"
    out_dir: str = "/content/drive/MyDrive/ChemDFM/results_nb02"
    split_col: str = "split_ho_pathway"
    model_tag: str = "residual_cellaware_mmd"
    pca_dim: int = 25
    batch_size: int = 512
    condition_col: str = "condition"
    cell_col: str = "cell_type"
    dose_col: str = "dose"
    fallback_dose_col: str = "dose_val"
cfg = CFG()
os.makedirs(cfg.out_dir, exist_ok=True)

Using device: cuda


In [ ]:
adata = ad.read_h5ad(cfg.data_path)
if cfg.dose_col not in adata.obs.columns and cfg.fallback_dose_col in adata.obs.columns:
    adata.obs[cfg.dose_col] = adata.obs[cfg.fallback_dose_col]

def map_split(x: str) -> str:
    x = str(x).lower()
    if "train" in x:
        return "train"
    if "test" in x or "val" in x:
        return "test"
    if "ood" in x:
        return "ood"
    return "drop"

adata.obs["_split3"] = adata.obs[cfg.split_col].astype(str).map(map_split)
adata = adata[adata.obs["_split3"].isin(["train","test","ood"])].copy()

X = adata.X
if hasattr(X, "toarray"):
    X = X.toarray()
X = np.asarray(X, dtype=np.float32)

train_mask = (adata.obs["_split3"] == "train").values
pca = PCA(n_components=cfg.pca_dim, random_state=SEED)
X_pca = np.zeros((adata.n_obs, cfg.pca_dim), dtype=np.float32)
X_pca[train_mask] = pca.fit_transform(X[train_mask]).astype(np.float32)
X_pca[~train_mask] = pca.transform(X[~train_mask]).astype(np.float32)

drug_enc = LabelEncoder()
cell_enc = LabelEncoder()
drug_enc.fit(adata.obs[cfg.condition_col].astype(str))
cell_enc.fit(adata.obs[cfg.cell_col].astype(str))
adata.obs["drug_idx"] = drug_enc.transform(adata.obs[cfg.condition_col].astype(str))
adata.obs["cell_idx"] = cell_enc.transform(adata.obs[cfg.cell_col].astype(str))
idx2cell = {i:c for i,c in enumerate(cell_enc.classes_)}

control_mask = (adata.obs[cfg.condition_col].astype(str) == "control").values
ctrl_means_pca, ctrl_means_gene = {}, {}
for cell in sorted(adata.obs[cfg.cell_col].astype(str).unique()):
    m = control_mask & (adata.obs[cfg.cell_col].astype(str).values == cell)
    ctrl_means_pca[cell] = X_pca[m].mean(axis=0).astype(np.float32)
    ctrl_means_gene[cell] = X[m].mean(axis=0).astype(np.float32)

X0_pca = np.stack([ctrl_means_pca[c] for c in adata.obs[cfg.cell_col].astype(str).values]).astype(np.float32)
X0_gene = np.stack([ctrl_means_gene[c] for c in adata.obs[cfg.cell_col].astype(str).values]).astype(np.float32)

In [ ]:
class ChemResidualDataset(Dataset):
    def __init__(self, split):
        mask = (adata.obs["_split3"].values == split) & (adata.obs[cfg.condition_col].astype(str).values != "control")
        self.idxs = np.where(mask)[0]
    def __len__(self):
        return len(self.idxs)
    def __getitem__(self, i):
        idx = self.idxs[i]
        row = adata.obs.iloc[idx]
        dose = float(row[cfg.dose_col])
        dose = np.log1p(max(dose, 0.0))
        return {
            "idx": int(idx),
            "x_true": torch.tensor(X_pca[idx], dtype=torch.float32),
            "x0": torch.tensor(X0_pca[idx], dtype=torch.float32),
            "drug_idx": torch.tensor(int(row["drug_idx"]), dtype=torch.long),
            "cell_idx": torch.tensor(int(row["cell_idx"]), dtype=torch.long),
            "dose": torch.tensor([dose], dtype=torch.float32),
            "condition": str(row[cfg.condition_col]),
            "cell_type": str(row[cfg.cell_col]),
            "pathway": str(row["pathway"]) if "pathway" in adata.obs.columns else "NA",
        }

test_loader = DataLoader(ChemResidualDataset("test"), batch_size=cfg.batch_size, shuffle=False)
ood_loader  = DataLoader(ChemResidualDataset("ood"), batch_size=cfg.batch_size, shuffle=False)
print("Test/OOD perturbed:", len(test_loader.dataset), len(ood_loader.dataset))

Test/OOD perturbed: 49834 10559


In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dims, out_dim, dropout=0.1):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

class StructuredDoseEncoder(nn.Module):
    def __init__(self, out_dim=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1,16), nn.ReLU(), nn.Linear(16,out_dim))
    def forward(self, dose):
        return self.net(dose)

class ResidualDoseResponseModel(nn.Module):
    def __init__(self, latent_dim, n_drugs, n_cells, emb_dim=32, hidden=256, dose_hidden=32, dropout=0.1):
        super().__init__()
        self.drug_emb = nn.Embedding(n_drugs, emb_dim)
        self.cell_emb = nn.Embedding(n_cells, emb_dim)
        self.dose_enc = StructuredDoseEncoder(dose_hidden)
        self.ctrl_enc = MLP(latent_dim, [hidden, hidden], hidden, dropout)
        fusion_in = hidden + emb_dim + emb_dim + dose_hidden
        self.delta_head = MLP(fusion_in, [hidden, hidden], latent_dim, dropout)
    def forward(self, x0, drug_idx, cell_idx, dose):
        z0 = self.ctrl_enc(x0)
        zd = self.drug_emb(drug_idx)
        zc = self.cell_emb(cell_idx)
        zz = self.dose_enc(dose)
        z = torch.cat([z0, zd, zc, zz], dim=1)
        delta_hat = self.delta_head(z)
        xhat = x0 + delta_hat
        return delta_hat, xhat

model = ResidualDoseResponseModel(
    latent_dim=cfg.pca_dim,
    n_drugs=len(drug_enc.classes_),
    n_cells=len(cell_enc.classes_),
).to(DEVICE)

best_ckpt = f"{cfg.ckpt_dir}/best_{cfg.model_tag}.pt"
model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
model.eval()
print("Loaded:", best_ckpt)

Loaded: /content/drive/MyDrive/ChemDFM/checkpoints_nb01/best_residual_cellaware_mmd.pt


In [ ]:
def rowwise_pearson(a, b):
    vals = []
    for i in range(a.shape[0]):
        if np.std(a[i]) < 1e-8 or np.std(b[i]) < 1e-8:
            continue
        vals.append(pearsonr(a[i], b[i])[0])
    return float(np.mean(vals)) if vals else np.nan

def compute_metrics(pred, true, x0):
    out = {
        "r2_full": float(r2_score(true.reshape(-1), pred.reshape(-1))),
        "pearson_rowmean": rowwise_pearson(true, pred),
        "mse": float(np.mean((true - pred)**2)),
    }
    for k in [20, 50]:
        vals = []
        for i in range(true.shape[0]):
            idx = np.argsort(-np.abs(true[i]-x0[i]))[:k]
            vals.append(r2_score(true[i, idx], pred[i, idx]))
        out[f"r2_top{k}"] = float(np.mean(vals))
    return out

@torch.no_grad()
def infer_split(loader, split_name):
    rows = []
    pred_pca_all, true_pca_all, x0_pca_all = [], [], []
    pred_gene_all, true_gene_all, x0_gene_all = [], [], []

    for batch in loader:
        idxs = np.array(batch["idx"], dtype=int)
        x0 = batch["x0"].to(DEVICE)
        x_true = batch["x_true"].to(DEVICE)
        drug_idx = batch["drug_idx"].to(DEVICE)
        cell_idx = batch["cell_idx"].to(DEVICE)
        dose = batch["dose"].to(DEVICE)
        _, x_hat = model(x0, drug_idx, cell_idx, dose)

        x_hat_np = x_hat.cpu().numpy()
        x_true_np = x_true.cpu().numpy()
        x0_np = x0.cpu().numpy()
        xhat_gene = pca.inverse_transform(x_hat_np).astype(np.float32)
        xtrue_gene = X[idxs].astype(np.float32)
        x0gene = X0_gene[idxs].astype(np.float32)

        pred_pca_all.append(x_hat_np)
        true_pca_all.append(x_true_np)
        x0_pca_all.append(x0_np)
        pred_gene_all.append(xhat_gene)
        true_gene_all.append(xtrue_gene)
        x0_gene_all.append(x0gene)

        for j, idx in enumerate(idxs):
            obs = adata.obs.iloc[idx]
            rows.append({
                "idx": int(idx),
                "split": split_name,
                "cell_type": str(obs[cfg.cell_col]),
                "condition": str(obs[cfg.condition_col]),
                "pathway": str(obs["pathway"]) if "pathway" in adata.obs.columns else "NA",
                "dose": float(obs[cfg.dose_col]),
            })

    meta = pd.DataFrame(rows)
    return (
        meta,
        np.concatenate(pred_pca_all, axis=0),
        np.concatenate(true_pca_all, axis=0),
        np.concatenate(x0_pca_all, axis=0),
        np.concatenate(pred_gene_all, axis=0),
        np.concatenate(true_gene_all, axis=0),
        np.concatenate(x0_gene_all, axis=0),
    )

test_meta, test_pred_pca, test_true_pca, test_x0_pca, test_pred_gene, test_true_gene, test_x0_gene = infer_split(test_loader, "test")
ood_meta,  ood_pred_pca,  ood_true_pca,  ood_x0_pca,  ood_pred_gene,  ood_true_gene,  ood_x0_gene  = infer_split(ood_loader, "ood")

In [ ]:
final_overall_rows = []
for split_name, pred, true, x0 in [
    ("test", test_pred_pca, test_true_pca, test_x0_pca),
    ("ood",  ood_pred_pca,  ood_true_pca,  ood_x0_pca),
]:
    final_overall_rows.append({"split": split_name, **compute_metrics(pred, true, x0)})

final_overall_df = pd.DataFrame(final_overall_rows)

final_per_cell_rows = []
for split_name, meta, pred, true, x0 in [
    ("test", test_meta, test_pred_pca, test_true_pca, test_x0_pca),
    ("ood",  ood_meta,  ood_pred_pca,  ood_true_pca,  ood_x0_pca),
]:
    for cell in sorted(meta["cell_type"].unique()):
        m = meta["cell_type"].values == cell
        final_per_cell_rows.append({"split": split_name, "cell_type": cell, **compute_metrics(pred[m], true[m], x0[m])})

final_per_cell_df = pd.DataFrame(final_per_cell_rows)
print("Final overall metrics:")
print(final_overall_df)
print("\nFinal per-cell metrics:")
print(final_per_cell_df)

final_overall_df.to_csv(f"{cfg.out_dir}/final_overall_metrics_posthoc.csv", index=False)
final_per_cell_df.to_csv(f"{cfg.out_dir}/final_per_cell_metrics_posthoc.csv", index=False)

Final overall metrics:
  split   r2_full  pearson_rowmean       mse  r2_top20  r2_top50
0  test  0.634606         0.783411  0.342530  0.497648  0.575017
1   ood  0.611967         0.770630  0.361605  0.481558  0.553353

Final per-cell metrics:
  split cell_type   r2_full  pearson_rowmean       mse  r2_top20  r2_top50
0  test      A549  0.741193         0.866627  0.297522  0.649857  0.719378
1  test      K562  0.627028         0.802341  0.455835  0.521488  0.584454
2  test      MCF7  0.522885         0.732939  0.307502  0.410773  0.499266
3   ood      A549  0.708858         0.848455  0.333310  0.614604  0.685561
4   ood      K562  0.601613         0.786149  0.456951  0.496592  0.554373
5   ood      MCF7  0.507758         0.721040  0.327927  0.402669  0.482087


In [ ]:
def compute_deg_auprc(meta, pred_gene, true_gene, x0_gene, topk=50):
    rows = []
    true_delta = true_gene - x0_gene
    pred_delta = pred_gene - x0_gene
    for i in range(true_gene.shape[0]):
        true_abs = np.abs(true_delta[i])
        pred_abs = np.abs(pred_delta[i])
        y_true = np.zeros_like(true_abs, dtype=np.int32)
        idx = np.argsort(-true_abs)[:topk]
        y_true[idx] = 1
        auprc = average_precision_score(y_true, pred_abs)
        rows.append({
            "split": meta.iloc[i]["split"],
            "cell_type": meta.iloc[i]["cell_type"],
            "condition": meta.iloc[i]["condition"],
            "deg_auprc_topk": float(auprc),
        })
    return pd.DataFrame(rows)

def compute_deg_percentile(meta, pred_gene, true_gene, x0_gene, percentile=95):
    rows = []
    true_delta = true_gene - x0_gene
    pred_delta = pred_gene - x0_gene
    for i in range(true_gene.shape[0]):
        true_abs = np.abs(true_delta[i])
        pred_abs = np.abs(pred_delta[i])
        thr = np.percentile(true_abs, percentile)
        y_true = (true_abs >= thr).astype(int)
        if y_true.sum() < 5:
            continue
        auprc = average_precision_score(y_true, pred_abs)
        rows.append({
            "split": meta.iloc[i]["split"],
            "cell_type": meta.iloc[i]["cell_type"],
            "condition": meta.iloc[i]["condition"],
            "deg_auprc_percentile": float(auprc),
        })
    return pd.DataFrame(rows)

deg_topk_df = pd.concat([
    compute_deg_auprc(test_meta, test_pred_gene, test_true_gene, test_x0_gene, topk=50),
    compute_deg_auprc(ood_meta,  ood_pred_gene,  ood_true_gene,  ood_x0_gene,  topk=50),
], ignore_index=True)

deg_percentile_df = pd.concat([
    compute_deg_percentile(test_meta, test_pred_gene, test_true_gene, test_x0_gene, percentile=95),
    compute_deg_percentile(ood_meta,  ood_pred_gene,  ood_true_gene,  ood_x0_gene,  percentile=95),
], ignore_index=True)

deg_topk_summary = deg_topk_df.groupby(["split","cell_type"], as_index=False)["deg_auprc_topk"].mean()
deg_percentile_summary = deg_percentile_df.groupby(["split","cell_type"], as_index=False)["deg_auprc_percentile"].mean()

print("\nDEG top-k summary:")
print(deg_topk_summary)
print("\nDEG percentile summary:")
print(deg_percentile_summary)

deg_topk_summary.to_csv(f"{cfg.out_dir}/deg_auprc_topk_summary.csv", index=False)
deg_percentile_summary.to_csv(f"{cfg.out_dir}/deg_auprc_percentile_summary.csv", index=False)


DEG top-k summary:
  split cell_type  deg_auprc_topk
0   ood      A549        0.146197
1   ood      K562        0.119509
2   ood      MCF7        0.204051
3  test      A549        0.144889
4  test      K562        0.114254
5  test      MCF7        0.183481

DEG percentile summary:
  split cell_type  deg_auprc_percentile
0   ood      A549              0.273152
1   ood      K562              0.267844
2   ood      MCF7              0.291366
3  test      A549              0.293532
4  test      K562              0.252629
5  test      MCF7              0.275430


In [ ]:
def pathway_recovery(meta, pred_gene, true_gene):
    tmp = meta.copy()
    tmp["row_idx"] = np.arange(len(tmp))
    rows = []
    for keys, sub in tmp.groupby(["split","cell_type","pathway"]):
        idx = sub["row_idx"].values
        if len(idx) < 5:
            continue
        pred_mean = pred_gene[idx].mean(axis=0)
        true_mean = true_gene[idx].mean(axis=0)
        corr = pearsonr(pred_mean, true_mean)[0] if np.std(pred_mean) > 1e-8 and np.std(true_mean) > 1e-8 else np.nan
        rows.append({
            "split": keys[0],
            "cell_type": keys[1],
            "pathway": keys[2],
            "n_samples": int(len(idx)),
            "pathway_profile_corr": float(corr) if corr == corr else np.nan,
            "pathway_profile_r2": float(r2_score(true_mean, pred_mean)),
        })
    return pd.DataFrame(rows)

pathway_df = pd.concat([
    pathway_recovery(test_meta, test_pred_gene, test_true_gene),
    pathway_recovery(ood_meta,  ood_pred_gene,  ood_true_gene),
], ignore_index=True)

pathway_summary = pathway_df.groupby(["split","cell_type"], as_index=False)[["pathway_profile_corr","pathway_profile_r2"]].mean()
print("\nPathway recovery summary:")
print(pathway_summary)

pathway_df.to_csv(f"{cfg.out_dir}/pathway_recovery_groupwise.csv", index=False)
pathway_summary.to_csv(f"{cfg.out_dir}/pathway_recovery_summary.csv", index=False)


Pathway recovery summary:
  split cell_type  pathway_profile_corr  pathway_profile_r2
0   ood      A549              0.978647            0.955634
1   ood      K562              0.970880            0.935216
2   ood      MCF7              0.975591            0.951751
3  test      A549              0.994296            0.988480
4  test      K562              0.991392            0.982500
5  test      MCF7              0.997502            0.994939


## Interpretation guide

When you send outputs back, I will use these rules:

- **Strong**: overall + OOD are above strongest baseline
- **Hard-case fix**: K562 improves or stays stable
- **Biological validity**: pathway profile correlation remains high
- **Reviewer-proof**: DEG summaries are nontrivial and per-cell tables make sense
